# KG1 NVIDIA Nemotron - v46 ALL-IN-ONE

## Um script que faz TUDO (após o install + restart)

### Root-cause fix vs v46 SOLVER_SFT (que bugou com loss=0.000000):
- **`use_dora=False`** (v30 PROVEN sem DoRA; PEFT #2274 quebra gradient em Mamba-2)
- Callback FAIL se loss < 0.5 (captura qualquer bug de masking futuro)
- pad_token != eos_token (secundário)
- max_length=1024 (samples avg 271 tok; 4096 era desperdício 93%)
- max_grad_norm=1.0 (NaN prevention)
- Smoke test + pre-score gate obrigatórios antes do treino real

### Estrutura:
- **Cell 1 (Install)**: torch 2.6 + mamba-ssm + causal-conv1d (RESTART obrigatório depois)
- **Cell 2 (Main)**: UM script que faz tudo (data + model + LoRA + train + auto-submit)

### Hardware: H100 HighRAM 80GB (BF16 full) ou A100 HighRAM 40GB (NF4 auto)

### Expected score: 0.72-0.78 (v30 baseline + solver-augmented 6540 CoTs)


In [ ]:
#@title CELL 1: Install (auto-detects GPU: Blackwell sm_100+ or Ampere/Hopper)
#@markdown ### APOS rodar: Runtime > Restart runtime, depois rode Cell 2
#@markdown ### Skipa se ja esta instalado (safe para re-run)

import subprocess, sys, os

def _sh(cmd):
    """Run shell, print last 1500 chars of stdout/stderr."""
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout or "") + (r.stderr or "")
    if out.strip():
        print(out[-1500:])
    return r.returncode

def _pkg_ver(name):
    try:
        import importlib.metadata
        return importlib.metadata.version(name)
    except Exception:
        return None

# ============================================================
# 0. Python version sanity
# ============================================================
py_maj_min = f"{sys.version_info.major}.{sys.version_info.minor}"
py_cpver = f"cp{sys.version_info.major}{sys.version_info.minor}"
print(f"Python: {py_maj_min} ({py_cpver})")
assert sys.version_info >= (3, 10), f"Python {py_maj_min} too old, need 3.10+"
assert sys.version_info < (3, 14), f"Python {py_maj_min} not supported by mamba-ssm wheels"

# ============================================================
# 1. Detect GPU compute capability (CRITICAL for torch version selection)
# ============================================================
print("\n=== GPU DETECT ===")
def _gpu_compute_cap():
    try:
        r = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=10,
        )
        if r.returncode != 0:
            return None, 0.0
        line = r.stdout.strip().split("\n")[0]
        name, cap = [x.strip() for x in line.split(",")]
        return name, float(cap)
    except Exception as e:
        print(f"  [WARN] nvidia-smi failed: {e}")
        return None, 0.0

gpu_name, gpu_cap = _gpu_compute_cap()
print(f"  GPU: {gpu_name}")
print(f"  Compute capability: sm_{int(gpu_cap*10) if gpu_cap else '?'}")

# Blackwell (sm_100, sm_101, sm_103, sm_120) requires CUDA 12.8+ / torch 2.7+
# Hopper (sm_90), Ampere (sm_80, sm_86) work with CUDA 12.4 / torch 2.6
IS_BLACKWELL = gpu_cap >= 10.0

if IS_BLACKWELL:
    PROFILE = "blackwell"
    TORCH_PIN = "torch==2.8.0"
    TORCH_INDEX = "https://download.pytorch.org/whl/cu128"
    TORCH_OK_PREFIX = "2.8."
    MAMBA_REL = "v2.3.1"
    MAMBA_FILE = f"mamba_ssm-2.3.1+cu12torch2.8cxx11abiTRUE-{py_cpver}-{py_cpver}-linux_x86_64.whl"
    CONV_REL = "v1.6.1.post4"
    CONV_FILE = f"causal_conv1d-1.6.1+cu12torch2.8cxx11abiTRUE-{py_cpver}-{py_cpver}-linux_x86_64.whl"
    print(f"  -> PROFILE: BLACKWELL (sm_{int(gpu_cap*10)}) requires torch 2.8 + cu128")
else:
    PROFILE = "ampere_hopper"
    TORCH_PIN = "torch==2.6.0"
    TORCH_INDEX = "https://download.pytorch.org/whl/cu124"
    TORCH_OK_PREFIX = "2.6."
    MAMBA_REL = "v2.2.4"
    MAMBA_FILE = f"mamba_ssm-2.2.4+cu12torch2.6cxx11abiFALSE-{py_cpver}-{py_cpver}-linux_x86_64.whl"
    CONV_REL = "v1.5.0.post8"
    CONV_FILE = f"causal_conv1d-1.5.0.post8+cu12torch2.6cxx11abiFALSE-{py_cpver}-{py_cpver}-linux_x86_64.whl"
    print(f"  -> PROFILE: AMPERE/HOPPER (sm_{int(gpu_cap*10)}) uses torch 2.6 + cu124")

CONV_URL = f"https://github.com/Dao-AILab/causal-conv1d/releases/download/{CONV_REL}/{CONV_FILE}"
MAMBA_URL = f"https://github.com/state-spaces/mamba/releases/download/{MAMBA_REL}/{MAMBA_FILE}"

# ============================================================
# 2. Check existing install vs profile (uninstall mismatched torch!)
# ============================================================
print("\n=== CHECKING EXISTING INSTALL ===")
torch_ver = _pkg_ver("torch")
mamba_ver = _pkg_ver("mamba-ssm") or _pkg_ver("mamba_ssm")
conv1d_ver = _pkg_ver("causal-conv1d") or _pkg_ver("causal_conv1d")
print(f"  torch:         {torch_ver}")
print(f"  mamba-ssm:     {mamba_ver}")
print(f"  causal-conv1d: {conv1d_ver}")
print(f"  required torch: {TORCH_OK_PREFIX}* ({PROFILE})")

torch_ok = torch_ver and torch_ver.startswith(TORCH_OK_PREFIX)
torch_wrong_ver = torch_ver and not torch_ok  # installed but wrong version

# CRITICAL: if torch is installed at WRONG version (e.g., 2.6 on Blackwell),
# we MUST uninstall and reinstall - mamba/conv1d wheels are torch-version-specific
if torch_wrong_ver:
    print(f"\n  [REINSTALL] torch {torch_ver} != {TORCH_OK_PREFIX}* - removing all torch+mamba+conv1d")
    _sh("pip uninstall -y torch torchvision torchaudio triton mamba-ssm mamba_ssm causal-conv1d causal_conv1d")
    torch_ok = False
    mamba_ver = None
    conv1d_ver = None

mamba_ok = bool(mamba_ver)
conv1d_ok = bool(conv1d_ver)

NEED_RESTART = False

if torch_ok and mamba_ok and conv1d_ok:
    print("\n  [OK] All critical packages installed at correct versions. No restart needed. Run Cell 2 directly.")
else:
    NEED_RESTART = True
    print("\n=== INSTALLING ===")

    # ---- Install torch (only torch needed - no torchvision/torchaudio for LoRA training) ----
    if not torch_ok:
        print(f"\n  Installing {TORCH_PIN} from {TORCH_INDEX} ...")
        rc = _sh(f"pip install --quiet {TORCH_PIN} --index-url {TORCH_INDEX}")
        if rc != 0:
            raise RuntimeError(f"FATAL: torch install failed (rc={rc}). Check pip output above.")

    # ---- Install causal-conv1d ----
    if not conv1d_ok:
        print(f"\n  Installing causal-conv1d ({CONV_REL}) ...")
        print(f"  URL: {CONV_URL}")
        if _sh(f"pip install --quiet {CONV_URL}") != 0:
            print("  [FAIL] causal-conv1d wheel install failed - Mamba-2 will use slow naive path")

    # ---- Install mamba-ssm ----
    if not mamba_ok:
        print(f"\n  Installing mamba-ssm ({MAMBA_REL}) ...")
        print(f"  URL: {MAMBA_URL}")
        if _sh(f"pip install --quiet {MAMBA_URL}") != 0:
            print("  [FAIL] mamba-ssm wheel install failed - will use slow naive path")

# ============================================================
# 3. Install other ML deps (no version pin - pip resolves)
# ============================================================
print("\n=== INSTALLING ML DEPS ===")
_sh("pip install --quiet -U transformers peft trl accelerate bitsandbytes datasets safetensors huggingface_hub kaggle")

# ============================================================
# 4. Final version check
# ============================================================
print("\n=== VERSION CHECK ===")
import importlib
all_ok = True
critical = {"torch": TORCH_OK_PREFIX, "mamba_ssm": None, "causal_conv1d": None,
            "transformers": None, "peft": None, "trl": None, "accelerate": None}
for pkg, expected_prefix in critical.items():
    try:
        # importlib.metadata uses dash, importlib.import_module uses underscore
        mod_name = pkg
        m = importlib.import_module(mod_name)
        v = getattr(m, "__version__", _pkg_ver(pkg.replace("_", "-")))
        marker = "[OK]"
        if expected_prefix and not (v and v.startswith(expected_prefix)):
            marker = "[FAIL]"
            all_ok = False
        print(f"  {marker} {pkg}: {v}")
    except Exception as e:
        print(f"  [FAIL] {pkg}: {e}")
        if expected_prefix or pkg in ("mamba_ssm", "causal_conv1d"):
            all_ok = False

print()
print(f"  PROFILE: {PROFILE}")
print(f"  GPU: {gpu_name} (sm_{int(gpu_cap*10)})")
print(f"  All critical OK: {all_ok}")

if NEED_RESTART:
    print("\n" + "=" * 60)
    print("  RESTART RUNTIME REQUIRED")
    print("  Click: Runtime > Restart session, then run Cell 2")
    print("=" * 60)
elif not all_ok:
    print("\n  [WARN] Some packages missing/wrong - check above before running Cell 2")
else:
    print("\n  [OK] Ready. Run Cell 2.")


In [ ]:
#@title CELL 2: ALL-IN-ONE (data + model + LoRA + train + auto-submit)
#@markdown ### Faz TUDO em um unico script. Rode esta celula apos Cell 1 + restart.

# ============================================================
# SECTION 0 - IMPORTS + VERSION GUARDS
# ============================================================
import os, sys, json, gc, re, math, time, random, glob, warnings
warnings.filterwarnings("ignore", category=UserWarning)

print("=" * 60)
print("KG1 v46 ALL-IN-ONE - Starting")
print("=" * 60)

import torch
import subprocess as _sp
print(f"\n[versions]")
print(f"  torch: {torch.__version__}")
assert torch.cuda.is_available(), "FATAL: CUDA not available"

# Detect GPU compute capability via nvidia-smi (works before any CUDA call)
try:
    _r = _sp.run(
        ["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=10,
    )
    _line = _r.stdout.strip().split("\n")[0]
    _gpu_name_early, _gpu_cap_str = [x.strip() for x in _line.split(",", 1)]
    _gpu_cap = float(_gpu_cap_str)
except Exception as _e:
    print(f"  [WARN] nvidia-smi failed: {_e}")
    _gpu_name_early, _gpu_cap = "unknown", 0.0

IS_BLACKWELL = _gpu_cap >= 10.0
print(f"  GPU: {_gpu_name_early} (sm_{int(_gpu_cap*10)})")

# Torch version must match GPU architecture
if IS_BLACKWELL:
    assert torch.__version__.startswith(("2.7.", "2.8.", "2.9.")), (
        f"FATAL: Blackwell GPU (sm_{int(_gpu_cap*10)}) requires torch 2.7+/cu128 "
        f"(got {torch.__version__}). Re-run Cell 1 (auto-detects) then RESTART runtime."
    )
else:
    assert torch.__version__.startswith(("2.6.", "2.7.", "2.8.")), (
        f"FATAL: torch {torch.__version__} not supported. Run Cell 1 then restart runtime."
    )

# CRITICAL: test CUDA kernel BEFORE loading 30B model (catches sm mismatch in <1s)
print("  [test] CUDA kernel sanity check ...")
try:
    _t = torch.zeros(64, 64, device="cuda", dtype=torch.bfloat16)
    _ = (_t @ _t).sum().item()
    del _t
    torch.cuda.empty_cache()
    print("  [OK] CUDA kernel works on this GPU")
except RuntimeError as _e:
    raise RuntimeError(
        f"FATAL: CUDA kernel test failed on {_gpu_name_early} (sm_{int(_gpu_cap*10)}).\n"
        f"Error: {_e}\n"
        f"This means torch {torch.__version__} was NOT built with kernels for your GPU.\n"
        f"FIX: Re-run Cell 1 (auto-detects GPU + installs right torch) then RESTART runtime."
    ) from _e

import numpy as np
import pandas as pd
from collections import Counter

import transformers, peft, trl, datasets
print(f"  transformers: {transformers.__version__}")
print(f"  peft: {peft.__version__}")
print(f"  trl: {trl.__version__}")
print(f"  datasets: {datasets.__version__}")

# Try import mamba (optional - will use naive path if missing)
try:
    import mamba_ssm
    print(f"  mamba_ssm: {getattr(mamba_ssm, '__version__', 'unknown')} [FAST PATH]")
except Exception:
    print(f"  mamba_ssm: NOT AVAILABLE [NAIVE PATH - 3-5x slower]")

try:
    import causal_conv1d
    print(f"  causal_conv1d: {getattr(causal_conv1d, '__version__', 'unknown')} [FAST]")
except Exception:
    print(f"  causal_conv1d: NOT AVAILABLE")

from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    TrainerCallback,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
from huggingface_hub import HfApi, hf_hub_download, create_repo, login

# ============================================================
# SECTION 1 - CONFIG
# ============================================================
CFG = {
    # Model
    "model_name": "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16",
    "data_repo": "felipesp1983/kg1-nemotron-training",
    "run_tag": "v46-allinone",

    # LoRA (v30 PROVEN: alpha=16, NO DoRA - fixes loss=0 root cause)
    "lora_rank": 32,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "use_dora": False,  # CRITICAL FIX: was True, broke Mamba-2 gradient (PEFT #2274)
    "target_modules": [
        "in_proj",                               # Mamba-2 input (forward() called, safe)
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention
        "gate_proj", "up_proj", "down_proj",     # MLP / MoE
        # NOTE: out_proj EXCLUDED (Mamba-2 broken per PEFT #2274)
    ],

    # Training (Multi-AI consensus: lr=7e-5, grad_accum=32, 5 epochs)
    "learning_rate": 7e-5,
    "n_epochs": 5,
    "grad_accum": 32,
    "warmup_ratio": 0.08,
    "max_length": 1024,       # FIX: was 4096, samples avg 271 tok, max 430
    "max_grad_norm": 1.0,     # NEW: NaN prevention
    "seed": 123,

    # Auto-submit schedule (6540 / 32 = ~204 steps/epoch, 5 epochs = ~1020 steps)
    "submit_steps": [100, 204, 408, 612, 1019],

    # Safety
    "freeze_moe_router": True,
    "smoke_min_loss": 0.5,    # FAIL if loss < this (masking bug)
    "smoke_max_loss": 20.0,   # FAIL if loss > this (LR too high or model broken)
}

OUTPUT_DIR = "/content/lora_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"\n[config]")
for k, v in CFG.items():
    print(f"  {k}: {v}")

# Reproducibility
random.seed(CFG["seed"])
np.random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])
torch.cuda.manual_seed_all(CFG["seed"])

# ============================================================
# SECTION 2 - AUTH (HF_KEY + KAGGLE)
# ============================================================
print("\n=== AUTH ===")

def _get_secret(*names):
    """Try Colab userdata first, then env vars."""
    try:
        from google.colab import userdata
        for n in names:
            v = userdata.get(n)
            if v:
                return v
    except Exception:
        pass
    for n in names:
        v = os.environ.get(n)
        if v:
            return v
    return None

HF_TOKEN = _get_secret("HF_KEY", "HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("FATAL: HF_KEY not in Colab Secrets (sidebar > Secrets > Add new secret)")
login(token=HF_TOKEN, add_to_git_credential=False)
os.environ["HF_TOKEN"] = HF_TOKEN
print(f"  [OK] HF login")

K_USER = _get_secret("KAGGLE_USERNAME")
K_KEY = _get_secret("KAGGLE_KEY")
KAGGLE_READY = False
if K_USER and K_KEY:
    os.environ["KAGGLE_USERNAME"] = K_USER
    os.environ["KAGGLE_KEY"] = K_KEY
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
        json.dump({"username": K_USER, "key": K_KEY}, f)
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    KAGGLE_READY = True
    print(f"  [OK] Kaggle: {K_USER}")
else:
    print(f"  [WARN] Kaggle secrets missing - manual submit only (HF upload still works)")

# ============================================================
# SECTION 3 - DOWNLOAD DATA
# ============================================================
print("\n=== DOWNLOAD DATA ===")
hf_hub_download(
    repo_id=CFG["data_repo"],
    filename="data/solver_augmented_train.jsonl",
    local_dir="/tmp/kg1_data",
    repo_type="dataset",
)
SOLVER_JSONL = "/tmp/kg1_data/data/solver_augmented_train.jsonl"

raw_rows = []
with open(SOLVER_JSONL, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            raw_rows.append(json.loads(line))
print(f"  [OK] loaded {len(raw_rows)} rows")
assert len(raw_rows) >= 6000, f"FATAL: only {len(raw_rows)} rows, expected >=6000"

# Filter answer length <= 24 (Kaggle DATA_GATE_POLICY)
filtered_rows = [r for r in raw_rows if len(str(r.get("answer", ""))) <= 24]
print(f"  After ans_len<=24: {len(filtered_rows)}")

FAMILY_MAP = {
    "gravity_constant": "grav",
    "unit_conversion": "unit",
    "numeral_system": "num",
    "text_encryption": "enc",
    "bit_manipulation": "bit",
    "equation_transform": "eq",
}
for r in filtered_rows:
    r["fam"] = FAMILY_MAP.get(r.get("family", ""), "other")

fam_counts = Counter(r["fam"] for r in filtered_rows)
print(f"  Family distribution:")
for fam in sorted(fam_counts):
    print(f"    {fam:6s}: {fam_counts[fam]:4d}")

random.shuffle(filtered_rows)

# ============================================================
# SECTION 4 - TOKENIZER + PAD TOKEN FIX
# ============================================================
print("\n=== TOKENIZER ===")
tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"], trust_remote_code=True)

print(f"  Before: pad={tokenizer.pad_token!r}(id={tokenizer.pad_token_id}), eos={tokenizer.eos_token!r}(id={tokenizer.eos_token_id})")

# CRITICAL: pad_token must differ from eos_token (else EOS gets -100 masked)
if tokenizer.pad_token is None or tokenizer.pad_token_id == tokenizer.eos_token_id:
    # Try reserved special tokens first
    vocab = tokenizer.get_vocab()
    candidates = ["<|pad|>", "<|extra_0|>", "<|reserved_0|>", "<|unused_0|>", "<unk>"]
    picked = None
    for c in candidates:
        if c in vocab and vocab[c] != tokenizer.eos_token_id:
            picked = c
            break
    if picked:
        tokenizer.pad_token = picked
        print(f"  Using reserved token as pad: {picked!r}")
    else:
        # Last resort: add a new pad token (requires model resize)
        tokenizer.add_special_tokens({"pad_token": "<|PAD|>"})
        print(f"  Added new pad_token <|PAD|> (will resize embeddings)")

print(f"  After:  pad={tokenizer.pad_token!r}(id={tokenizer.pad_token_id}), eos={tokenizer.eos_token!r}(id={tokenizer.eos_token_id})")
assert tokenizer.pad_token_id != tokenizer.eos_token_id, "FATAL: pad_token_id must differ from eos_token_id"

# ============================================================
# SECTION 5 - BUILD DATASET
# ============================================================
print("\n=== BUILD DATASET ===")
PROMPT_SUFFIX = "\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`"

def extract_reasoning(content, fallback):
    """Strip trailing \\boxed{...} from assistant content."""
    m = re.search(r"\\boxed\{([^}]*)\}\s*$", content)
    if m:
        return content[:m.start()].rstrip(), m.group(1)
    return content.rstrip(), fallback

examples = []
skipped = 0
for row in filtered_rows:
    prompt = str(row["prompt"])
    row_answer = str(row["answer"])
    msgs = row.get("messages", [])
    asst_msg = next((m for m in msgs if m.get("role") == "assistant"), None)
    if not asst_msg:
        skipped += 1
        continue
    orig = asst_msg.get("content", "")
    if not orig.strip():
        skipped += 1
        continue
    reasoning, _ = extract_reasoning(orig, row_answer)
    if len(reasoning.strip()) < 5:
        reasoning = orig.rstrip()

    asst_content = f"<think>\n{reasoning}\n</think>\n\\boxed{{{row_answer}}}"
    user_content = prompt + PROMPT_SUFFIX
    examples.append({
        "messages": [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": asst_content},
        ]
    })

print(f"  Built {len(examples)} examples (skipped {skipped})")
assert len(examples) >= 5000, f"FATAL: only {len(examples)} examples, expected >=5000"

# Apply chat template -> single text field
texts = []
for ex in examples:
    text = tokenizer.apply_chat_template(
        ex["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    texts.append(text)

ds = Dataset.from_dict({"text": texts})
print(f"  Dataset: {len(ds)} examples")

# Token length stats
lens = [len(tokenizer.encode(t)) for t in texts[:300]]
print(f"  Token lengths (first 300): avg={np.mean(lens):.0f}, max={max(lens)}, p95={np.percentile(lens,95):.0f}, p99={np.percentile(lens,99):.0f}")
if max(lens) > CFG["max_length"]:
    n_trunc = sum(1 for l in lens if l > CFG["max_length"])
    print(f"  [WARN] {n_trunc}/300 samples exceed max_length={CFG['max_length']}")

print(f"\n  Sample[0] (first 600 chars):\n{texts[0][:600]}")

# ============================================================
# SECTION 6 - GPU MODE SELECTION (NF4 vs BF16 full)
# ============================================================
print("\n=== GPU MODE ===")
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
gpu_name = torch.cuda.get_device_name(0)
print(f"  GPU: {gpu_name}")
print(f"  VRAM: {gpu_mem_gb:.1f} GB")
print(f"  Compute: sm_{int(_gpu_cap*10)} ({'BLACKWELL' if IS_BLACKWELL else 'AMPERE/HOPPER'})")

# Mode: VRAM-based (40GB needs NF4, 70GB+ does BF16 full)
USE_NF4 = gpu_mem_gb < 70
_mode_label = "NF4 4-bit (low VRAM fallback)" if USE_NF4 else "BF16 full precision"
print(f"  Mode: {_mode_label}")

# ============================================================
# SECTION 7 - LOAD MODEL
# ============================================================
print("\n=== LOAD MODEL (takes 5-8 min) ===")

if USE_NF4:
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        CFG["model_name"],
        quantization_config=bnb,
        device_map={"": 0},
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
    )
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
else:
    model = AutoModelForCausalLM.from_pretrained(
        CFG["model_name"],
        device_map={"": 0},
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
    )
    model.gradient_checkpointing_enable()

print(f"  [OK] model loaded. dtype={next(model.parameters()).dtype}")

# Resize if we added a new pad token
emb_size = model.get_input_embeddings().weight.shape[0]
if len(tokenizer) > emb_size:
    print(f"  Resizing embeddings: {emb_size} -> {len(tokenizer)}")
    model.resize_token_embeddings(len(tokenizer))

# ============================================================
# SECTION 8 - MoE CLASS-LEVEL PATCH (dtype fix)
# ============================================================
print("\n=== MoE PATCH ===")
_patched = []
for mod_name, mod in list(sys.modules.items()):
    if "nemotron_h" not in mod_name.lower():
        continue
    for attr_name in dir(mod):
        try:
            cls = getattr(mod, attr_name, None)
        except Exception:
            continue
        if not isinstance(cls, type):
            continue
        name_upper = attr_name.upper()
        if "MOE" in name_upper and hasattr(cls, "forward"):
            orig = cls.forward
            def _patched_forward(self, hidden_states, *args, _orig=orig, **kwargs):
                out = _orig(self, hidden_states, *args, **kwargs)
                if isinstance(out, tuple):
                    return tuple(
                        x.to(hidden_states.dtype) if isinstance(x, torch.Tensor) else x
                        for x in out
                    )
                return out.to(hidden_states.dtype) if isinstance(out, torch.Tensor) else out
            cls.forward = _patched_forward
            _patched.append(f"{mod_name}.{attr_name}")

print(f"  Patched classes: {_patched if _patched else 'NONE (relying on BF16 path)'}")

# Freeze MoE routers (stability)
if CFG["freeze_moe_router"]:
    frozen = 0
    for n, p in model.named_parameters():
        if "router" in n.lower():
            p.requires_grad = False
            frozen += 1
    print(f"  Frozen {frozen} MoE router params")

# ============================================================
# SECTION 9 - APPLY LoRA (use_dora=False - ROOT CAUSE FIX)
# ============================================================
print("\n=== APPLY LoRA ===")
lora_config = LoraConfig(
    r=CFG["lora_rank"],
    lora_alpha=CFG["lora_alpha"],
    lora_dropout=CFG["lora_dropout"],
    target_modules=CFG["target_modules"],
    bias="none",
    task_type="CAUSAL_LM",
    use_dora=CFG["use_dora"],  # FALSE - v30 proven, fixes loss=0
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Enable grads on LoRA
for n, p in model.named_parameters():
    if "lora_" in n:
        p.requires_grad = True

# ============================================================
# SECTION 10 - SMOKE TEST (2 steps, strict gate)
# ============================================================
print("\n=== SMOKE TEST (2 steps) ===")

smoke_args = SFTConfig(
    output_dir="/tmp/smoke_out",
    num_train_epochs=1,
    max_steps=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=CFG["learning_rate"],
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    bf16=True,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    dataset_text_field="text",
    max_length=CFG["max_length"],
    max_grad_norm=CFG["max_grad_norm"],
    seed=CFG["seed"],
)

smoke_trainer = SFTTrainer(
    model=model,
    train_dataset=ds.select(range(min(8, len(ds)))),
    processing_class=tokenizer,
    args=smoke_args,
)
smoke_trainer.train()

# Extract losses
smoke_losses = [
    float(l["loss"]) for l in smoke_trainer.state.log_history
    if "loss" in l and l["loss"] is not None
]
print(f"\n  Smoke losses: {smoke_losses}")

assert smoke_losses, "SMOKE FAIL: no losses logged"
last_loss = smoke_losses[-1]
assert not math.isnan(last_loss) and not math.isinf(last_loss), f"SMOKE FAIL: loss={last_loss}"
assert last_loss >= CFG["smoke_min_loss"], f"SMOKE FAIL: loss={last_loss:.4f} < {CFG['smoke_min_loss']} (likely masking/DoRA bug)"
assert last_loss <= CFG["smoke_max_loss"], f"SMOKE FAIL: loss={last_loss:.4f} > {CFG['smoke_max_loss']} (LR too high or model broken)"

print(f"  [OK] SMOKE PASS: loss={last_loss:.4f} in [{CFG['smoke_min_loss']}, {CFG['smoke_max_loss']}]")

# Cleanup
del smoke_trainer
gc.collect()
torch.cuda.empty_cache()

# ============================================================
# SECTION 11 - REAL TRAINING with CALLBACK
# ============================================================
print("\n=== REAL TRAINING ===")

HF_REPO_ID = f"felipesp1983/kg1-nemotron-lora-{CFG['run_tag']}"
hf_api = HfApi(token=HF_TOKEN)

class GuardedSubmitCallback(TrainerCallback):
    def __init__(self, submit_steps, output_dir, hf_repo_id, api, hf_token):
        self.submit_steps = set(submit_steps)
        self.output_dir = output_dir
        self.hf_repo_id = hf_repo_id
        self.api = api
        self.hf_token = hf_token
        self.uploaded = set()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        loss = logs.get("loss")
        step = state.global_step
        if not isinstance(loss, (int, float)):
            return

        # GUARD 1: NaN/Inf
        if math.isnan(loss) or math.isinf(loss):
            print(f"\n!!! STOP: NaN/Inf loss at step {step}")
            control.should_training_stop = True
            return

        # GUARD 2: loss=0 (masking/DoRA bug)
        if loss < CFG["smoke_min_loss"] and step > 2:
            print(f"\n!!! STOP: loss={loss:.4f} < {CFG['smoke_min_loss']} at step {step} (masking bug)")
            control.should_training_stop = True
            return

        # GUARD 3: explosion (>30)
        if loss > 30.0 and step > 5:
            print(f"\n!!! STOP: loss={loss:.2f} > 30 at step {step} (explosion)")
            control.should_training_stop = True
            return

        # GUARD 4: step 10 calibration
        if step == 10:
            if loss > 25.0:
                print(f"\n!!! WARN step 10 loss={loss:.2f} very high - monitoring")
            elif loss > 20.0:
                print(f"\n[WARN] step 10 loss={loss:.2f} high")
            else:
                print(f"\n[OK] step 10 loss={loss:.2f} healthy")

    def on_save(self, args, state, control, **kwargs):
        step = state.global_step
        if step not in self.submit_steps or step in self.uploaded:
            return
        self.uploaded.add(step)
        ckpt_dir = os.path.join(self.output_dir, f"checkpoint-{step}")
        if not os.path.isdir(ckpt_dir):
            print(f"\n[WARN] checkpoint {ckpt_dir} not found")
            return
        repo_id = f"{self.hf_repo_id}-step{step}"
        print(f"\n=== AUTO-UPLOAD checkpoint-{step} -> {repo_id} ===")
        try:
            create_repo(repo_id, token=self.hf_token, repo_type="model", exist_ok=True, private=True)
            self.api.upload_folder(
                folder_path=ckpt_dir,
                repo_id=repo_id,
                repo_type="model",
                token=self.hf_token,
            )
            print(f"  [OK] uploaded to {repo_id}")
        except Exception as e:
            print(f"  [ERROR] upload failed: {e}")

train_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=CFG["n_epochs"],
    per_device_train_batch_size=1,
    gradient_accumulation_steps=CFG["grad_accum"],
    learning_rate=CFG["learning_rate"],
    warmup_ratio=CFG["warmup_ratio"],
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=10,
    report_to="none",
    bf16=True,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    dataset_text_field="text",
    max_length=CFG["max_length"],
    max_grad_norm=CFG["max_grad_norm"],
    seed=CFG["seed"],
    optim="adamw_torch",
    dataloader_num_workers=0,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=ds,
    processing_class=tokenizer,
    args=train_args,
    callbacks=[GuardedSubmitCallback(
        submit_steps=CFG["submit_steps"],
        output_dir=OUTPUT_DIR,
        hf_repo_id=HF_REPO_ID,
        api=hf_api,
        hf_token=HF_TOKEN,
    )],
)

total_steps = (len(ds) * CFG["n_epochs"]) // CFG["grad_accum"]
print(f"\n  Dataset: {len(ds)}")
print(f"  Epochs: {CFG['n_epochs']}")
print(f"  Total optim steps: ~{total_steps}")
print(f"  Submit at: {CFG['submit_steps']}")
print(f"  LR: {CFG['learning_rate']}, max_length: {CFG['max_length']}, DoRA: {CFG['use_dora']}")
print(f"\n  Starting training...")

trainer.train()

# ============================================================
# SECTION 12 - SAVE + UPLOAD FINAL
# ============================================================
print("\n=== SAVE FINAL ===")
final_dir = os.path.join(OUTPUT_DIR, "final")
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
print(f"  [OK] saved to {final_dir}")

try:
    create_repo(HF_REPO_ID, token=HF_TOKEN, repo_type="model", exist_ok=True, private=True)
    hf_api.upload_folder(
        folder_path=final_dir,
        repo_id=HF_REPO_ID,
        repo_type="model",
        token=HF_TOKEN,
    )
    print(f"  [OK] uploaded final to {HF_REPO_ID}")
except Exception as e:
    print(f"  [ERROR] upload final: {e}")

print("\n" + "=" * 60)
print("CELL 2 COMPLETE")
print("=" * 60)
print(f"  Checkpoints: {OUTPUT_DIR}")
print(f"  HF repo:     {HF_REPO_ID}")
print(f"  Next steps: strip_moe_experts + Kaggle submit (see memory/reference_submission_format.md)")
